In [2]:
import pandas as pd
import networkx as nx
import numpy as np
from pathlib import Path
from node2vec import Node2Vec
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
data_dir = Path("./")

df_friend = pd.read_csv(data_dir / "friend_with.csv")

print("Shape:", df_friend.shape)
print("Columns:", df_friend.columns.tolist())
display(df_friend.head(10))

Shape: (10442, 3)
Columns: ['startUserId', 'endUserId', 'since']


,startUserId,endUserId,since
0,U0001,U0129,2025-05-03T01:10:55
1,U0129,U0001,2025-05-03T01:10:55
2,U0001,U0748,2026-01-16T22:01:53
3,U0748,U0001,2026-01-16T22:01:53
4,U0001,U0673,2025-09-05T14:45:18
5,U0673,U0001,2025-09-05T14:45:18
6,U0001,U0276,2026-03-16T17:37:21
7,U0276,U0001,2026-03-16T17:37:21
8,U0001,U0409,2025-07-03T03:47:54
9,U0409,U0001,2025-07-03T03:47:54


In [4]:
SOURCE_COL = "startUserId"
TARGET_COL = "endUserId"

In [5]:
df_edges = df_friend[[SOURCE_COL, TARGET_COL]].copy()

# bỏ null
df_edges = df_edges.dropna()

# bỏ self-loop
df_edges = df_edges[df_edges[SOURCE_COL] != df_edges[TARGET_COL]]

# chuẩn hóa cạnh vô hướng: (A,B) và (B,A) thành 1
df_edges["user_min"] = df_edges[[SOURCE_COL, TARGET_COL]].min(axis=1)
df_edges["user_max"] = df_edges[[SOURCE_COL, TARGET_COL]].max(axis=1)

df_edges = df_edges[["user_min", "user_max"]].drop_duplicates()
df_edges.columns = ["source", "target"]

print("Số cạnh sau làm sạch:", len(df_edges))
display(df_edges.head(10))

Số cạnh sau làm sạch: 5221


,source,target
0,U0001,U0129
2,U0001,U0748
4,U0001,U0673
6,U0001,U0276
8,U0001,U0409
10,U0001,U0296
12,U0001,U0544
14,U0002,U0085
16,U0002,U0523
18,U0002,U0181


In [6]:
G = nx.from_pandas_edgelist(
    df_edges,
    source="source",
    target="target",
    create_using=nx.Graph()
)

print("Số node:", G.number_of_nodes())
print("Số edge:", G.number_of_edges())

Số node: 900
Số edge: 5221


In [7]:
from node2vec import Node2Vec
import os

node2vec = Node2Vec(
    G,
    dimensions=128,     # tăng độ giàu thông tin embedding
    walk_length=30,     # mỗi walk dài hơn
    num_walks=50,      # từ 50 -> 200 (nặng hơn rõ rệt)
    workers=4,
    p=1,
    q=2,
    seed=42
)

model = node2vec.fit(
    window=10,
    min_count=1,
    batch_words=128,
    epochs=10           # tăng số epoch Word2Vec
)

print("Train xong Node2Vec")


Computing transition probabilities:   0%|          | 0/900 [00:00<?, ?it/s]

Train xong Node2Vec


In [9]:
nodes = list(G.nodes())
embeddings = np.array([model.wv[str(n)] for n in nodes])
print("Embedding shape:", embeddings.shape)


Embedding shape: (900, 128)


In [10]:
# KMeans trên embedding Node2Vec

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd
import numpy as np

# embeddings: đã có từ notebook của bạn
# nodes: list user_id tương ứng từng dòng embedding

# 1) Chọn số cụm K tự động (thử từ 6..11)
k_range = range(6, 12)
best_k = None
best_score = -1

for k in k_range:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels_tmp = km_tmp.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels_tmp)
    print(f"k={k}, silhouette={score:.4f}")
    if score > best_score:
        best_score = score
        best_k = k

print(f"\nBest k = {best_k}, best silhouette = {best_score:.4f}")

# 2) Train KMeans với K tốt nhất
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(embeddings)

# 3) Gán cụm cho user
df_cluster = pd.DataFrame({
    "user_id": nodes,
    "cluster": cluster_labels
})

display(df_cluster.head(20))



c:\Users\Thinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


k=6, silhouette=0.1306
k=7, silhouette=0.1502


c:\Users\Thinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
c:\Users\Thinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


k=8, silhouette=0.1689


c:\Users\Thinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


...

k=10, silhouette=0.1319


c:\Users\Thinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


k=11, silhouette=0.1197

Best k = 8, best silhouette = 0.1689


c:\Users\Thinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


,user_id,cluster
0,U0001,4
1,U0129,4
2,U0748,4
3,U0673,4
4,U0276,4
5,U0409,4
6,U0296,4
7,U0544,4
8,U0002,3
9,U0085,3


In [12]:
fig.update_layout(
    title=f"3D Friend Graph - KMeans clusters (k={k})",
    showlegend=False,
    margin=dict(l=0, r=0, b=0, t=45),
    scene=dict(
        xaxis=dict(
            title="X",
            visible=True,
            showgrid=True,
            zeroline=True,
            showticklabels=True,
            backgroundcolor="rgb(245,245,245)"
        ),
        yaxis=dict(
            title="Y",
            visible=True,
            showgrid=True,
            zeroline=True,
            showticklabels=True,
            backgroundcolor="rgb(245,245,245)"
        ),
        zaxis=dict(
            title="Z",
            visible=True,
            showgrid=True,
            zeroline=True,
            showticklabels=True,
            backgroundcolor="rgb(245,245,245)"
        ),
        aspectmode="data"
    )
)
fig.show()


In [13]:
# Xem vector embedding của từng user (sau khi đã train model)

nodes = list(G.nodes())

# 1) In thử 5 user đầu
for u in nodes[:5]:
    vec = model.wv[str(u)]
    print(f"{u} | dim={len(vec)}")
    print(np.round(vec[:10], 4))  # in 10 chiều đầu cho gọn
    print("-" * 50)

# 2) Lấy vector của 1 user cụ thể
user_id = "U0001"
vec_u = model.wv[str(user_id)]
print(f"Vector của {user_id}:")
print(vec_u)  # full vector


U0001 | dim=128
[-0.2618  0.0244  0.36   -0.1357  0.3559 -0.2193 -0.323  -0.4102  0.0255
  0.0873]
--------------------------------------------------
U0129 | dim=128
[-0.2541 -0.2155  0.2621  0.1144  0.0275  0.0604 -0.1365  0.0054  0.5004
  0.0259]
--------------------------------------------------
U0748 | dim=128
[-0.2306 -0.084   0.4061  0.0978  0.2348 -0.3073 -0.3711 -0.2271 -0.1131
 -0.1088]
--------------------------------------------------
U0673 | dim=128
[-0.3395 -0.13    0.1447 -0.2179  0.2125 -0.0377 -0.2808 -0.2214 -0.1734
  0.1543]
--------------------------------------------------
U0276 | dim=128
[-0.5283  0.0892  0.2896  0.1365  0.3604 -0.1016 -0.0106 -0.2449 -0.0446
  0.3688]
--------------------------------------------------
Vector của U0001:
[-0.2617633   0.0243878   0.3600285  -0.13572447  0.35594618 -0.21934655
 -0.32298428 -0.4101838   0.02547682  0.08727728  0.14461437  0.05434782
  0.03948139 -0.02927976  0.25484085  0.08632797 -0.11074647  0.13786188
 -0.07116226 

In [14]:
# Tạo ma trận cosine similarity từ embedding Node2Vec

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# nodes: list user_id
# model: model Node2Vec đã train

nodes = list(G.nodes())
embeddings = np.array([model.wv[str(u)] for u in nodes])

# ma trận cosine (N x N)
sim_matrix = cosine_similarity(embeddings)

print("sim_matrix shape:", sim_matrix.shape)
print("Cosine(U0001, U0001) =", sim_matrix[nodes.index("U0001"), nodes.index("U0001")])

# Đưa về DataFrame cho dễ nhìn
df_sim = pd.DataFrame(sim_matrix, index=nodes, columns=nodes)
display(df_sim.iloc[:10, :10])  # xem góc 10x10




sim_matrix shape: (900, 900)
Cosine(U0001, U0001) = 1.0


,U0001,U0129,U0748,U0673,U0276,U0409,U0296,U0544,U0002,U0085
U0001,1.000000,0.580319,0.589388,0.627856,0.664442,0.698325,0.660410,0.652741,0.203094,0.163355
U0129,0.580319,1.000000,0.564485,0.436588,0.505740,0.487801,0.447016,0.487688,0.116913,0.176890
U0748,0.589388,0.564485,1.000000,0.495661,0.427245,0.425986,0.379514,0.385614,0.035776,0.127034
U0673,0.627856,0.436588,0.495661,1.000000,0.552612,0.594133,0.363538,0.515501,0.176506,0.107066
U0276,0.664442,0.505740,0.427245,0.552612,1.000000,0.633755,0.539127,0.700468,0.176743,0.104722
U0409,0.698325,0.487801,0.425986,0.594133,0.633755,1.000000,0.483695,0.579294,0.194319,0.128346
U0296,0.660410,0.447016,0.379514,0.363538,0.539127,0.483695,1.000000,0.562070,0.114165,0.142497
U0544,0.652741,0.487688,0.385614,0.515501,0.700468,0.579294,0.562070,1.000000,0.139017,0.138990
U0002,0.203094,0.116913,0.035776,0.176506,0.176743,0.194319,0.114165,0.139017,1.000000,0.634774
U0085,0.163355,0.176890,0.127034,0.107066,0.104722,0.128346,0.142497,0.138990,0.634774,1.000000
